# Chapter 11: Benchmarking, Evaluation, and When Classical ML Wins

Run the three printed listings in order. See Appendix IV for the tested base environment. The example uses synthetic data and a fixed classical SVC; circuit executions and shots are zero, and depth is not applicable. Fold SD is descriptive; the bootstrap interval conditions on the fitted model. Memory is peak traced allocations, not total process memory.

For the conceptual surrogate circuit-optimization protocol, see the chapter section on ML-assisted quantum circuit optimization.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=160, n_features=4, n_informative=3,
    n_redundant=0, n_clusters_per_class=2,
    weights=[0.5, 0.5], flip_y=0.01, class_sep=1.3,
    random_state=11,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=11
)
print("Train/test shapes:", X_train.shape, X_test.shape)
print("Class counts:", np.bincount(y_train), np.bincount(y_test))


In [ ]:
from time import perf_counter
import tracemalloc
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

baseline = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1.0, gamma="scale")),
])


def measured_fit_predict(train_x, train_y, eval_x, eval_y):
    """Fit fresh training-only preprocessing; return model, labels, costs."""
    tracemalloc.start()
    try:
        start = perf_counter()
        model = clone(baseline)
        model.fit(train_x, train_y)
        pred = model.predict(eval_x)
        seconds = perf_counter() - start
        _, peak_bytes = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()
    record = {
        "score": balanced_accuracy_score(eval_y, pred),
        "seconds": seconds,
        "peak_kib": peak_bytes / 1024,
    }
    return model, pred, record


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=11)
fold_records = []
for train_idx, valid_idx in cv.split(X_train, y_train):
    _, _, record = measured_fit_predict(
        X_train[train_idx], y_train[train_idx],
        X_train[valid_idx], y_train[valid_idx],
    )
    fold_records.append(record)

# Fixed settings: no hyperparameter search or test-driven changes.
final_model, test_pred, test_record = measured_fit_predict(
    X_train, y_train, X_test, y_test
)
for label, record in list(enumerate(fold_records, 1)) + [
    ("test", test_record)
]:
    print(label, {k: round(v, 4) for k, v in record.items()})


In [ ]:
validation_scores = np.array([r["score"] for r in fold_records])
print("CV mean/SD:", validation_scores.mean(),
      validation_scores.std(ddof=1))

# Resample paired outcomes within each true class; do not refit.
rng = np.random.default_rng(101)
classes = np.unique(y_test)
bootstrap_scores = np.zeros(2000)
for label in classes:
    correct = (test_pred[y_test == label] == label).astype(float)
    draws = rng.choice(correct, size=(2000, len(correct)), replace=True)
    bootstrap_scores += draws.mean(axis=1) / len(classes)
test_interval = np.quantile(bootstrap_scores, [0.025, 0.975])
print("Test score:", test_record["score"])
print("Conditional 95% bootstrap interval:", test_interval)

all_records = fold_records + [test_record]
resource_record = {
    "model_fits": len(all_records),
    "fit_predict_seconds": sum(r["seconds"] for r in all_records),
    "max_traced_peak_kib": max(r["peak_kib"] for r in all_records),
    "circuit_executions": 0,
    "shots": 0,
    "qubits": None,
    "circuit_depth": None,
    "two_qubit_gates": None,
}
print("Resources (None means not applicable):", resource_record)
